In [1]:
import datasets
import json
import pandas as pd
import numpy as np

In [2]:
from transformers import AutoTokenizer

# EDA for ManyTypes4Py

In [3]:
path = "../../data/ManyTypes4PyDataset-v0.7/"

In [4]:
df = pd.read_csv(path + "dataset_split.csv")

In [5]:
df

,train,repos/HackPartners/shush/setup.py
0,train,repos/HackPartners/shush/shush/client.py
1,train,repos/HackPartners/shush/shush/__init__.py
2,train,repos/tedle/uitabot/bot/uitabot.py
3,train,repos/tedle/uitabot/bot/test/test_audio.py
4,train,repos/tedle/uitabot/bot/test/test_youtube_api.py
...,...,...
293664,valid,repos/binaryshao/ST3config/Packages/Python PEP...
293665,valid,repos/binaryshao/ST3config/Packages/ConvertToU...
293666,valid,repos/binaryshao/ST3config/Packages/Anaconda/a...
293667,valid,repos/binaryshao/ST3config/Packages/Anaconda/a...


In [7]:
df.loc[0]

train                                                                   train
repos/HackPartners/shush/setup.py    repos/HackPartners/shush/shush/client.py
Name: 0, dtype: object

In [5]:
sample_path = "../../data/ManyTypes4PyDataset-v0.7/processed_projects_clean/sim1234snowid.json"

In [6]:
with open(sample_path, "rb") as f:
    sample = json.load(f)

In [19]:
source_code = sample['sim1234/snowid']['src_files']['repos/sim1234/snowid/project/game.py']['untyped_seq']

In [20]:
code = source_code.replace("[EOL] ", "\n").replace("[EOL]", "\n")

In [21]:
print(code)

from typing import Any , Type 
import typing 
import project 
import os 

import sdl2 
import sdl2 . ext 

from gamepart import Game 

from context import MyContext 


ROOT = os . path . dirname ( os . path . abspath ( __file__ ) ) 
RESOURCES = os . path . join ( ROOT , [string] ) 


class MyGame ( Game ) : 
context_class = MyContext 

def get_config ( self ) : 
config = super ( ) . get_config ( ) 
config [ [string] ] = [number] 
[comment] 
return config 

def init_font_manager ( self ) : 
self . font_manager = sdl2 . ext . FontManager ( os . path . join ( RESOURCES , [string] ) ) 

def init_heavy ( self ) : 
self . font_manager . add ( os . path . join ( RESOURCES , [string] ) , [string] ) 

def init_scenes ( self ) : 
import scenes 

self . add_scene ( [string] , scenes . TestScene ) 
self . add_scene ( [string] , scenes . BallScene ) 
super ( ) . init_scenes ( ) 
self . queue_scene_switch ( [string] ) 



In [22]:
len(source_code.split(" "))

220

In [23]:
source_types = sample['sim1234/snowid']['src_files']['repos/sim1234/snowid/project/game.py']['typed_seq']

In [31]:
code_types = source_types.split(" ")
for el, typ in zip(source_code.split(" "), source_types.split(" ")):
    if typ != "0":
        print(el, typ)

ROOT $builtins.str$
RESOURCES $builtins.str$
ROOT $builtins.str$
context_class $typing.Type[project.game.MyGame]$
config $typing.Any$
config $typing.Any$
config $typing.Any$
RESOURCES $builtins.str$
RESOURCES $builtins.str$


In [9]:
sample

{'sim1234/snowid': {'src_files': {'repos/sim1234/snowid/project/game.py': {'untyped_seq': 'from typing import Any , Type [EOL] import typing [EOL] import project [EOL] import os [EOL] [EOL] import sdl2 [EOL] import sdl2 . ext [EOL] [EOL] from gamepart import Game [EOL] [EOL] from context import MyContext [EOL] [EOL] [EOL] ROOT = os . path . dirname ( os . path . abspath ( __file__ ) ) [EOL] RESOURCES = os . path . join ( ROOT , [string] ) [EOL] [EOL] [EOL] class MyGame ( Game ) : [EOL] context_class = MyContext [EOL] [EOL] def get_config ( self ) : [EOL] config = super ( ) . get_config ( ) [EOL] config [ [string] ] = [number] [EOL] [comment] [EOL] return config [EOL] [EOL] def init_font_manager ( self ) : [EOL] self . font_manager = sdl2 . ext . FontManager ( os . path . join ( RESOURCES , [string] ) ) [EOL] [EOL] def init_heavy ( self ) : [EOL] self . font_manager . add ( os . path . join ( RESOURCES , [string] ) , [string] ) [EOL] [EOL] def init_scenes ( self ) : [EOL] import scenes 

# CUAD

In [3]:
# Prepare CUAD
# Check lengths distribution
# split by segments - from 1-2 to 8-16
# save splits

In [4]:
cuad = datasets.load_dataset("theatticusproject/cuad-qa")

Using the latest cached version of the module from /home/jovyan/.cache/huggingface/modules/datasets_modules/datasets/theatticusproject--cuad-qa/01ed7dc61ab84230462731422e77cbb6f54ea8590b22a2d881b594f4d7f3746c (last modified on Fri Apr 11 18:41:35 2025) since it couldn't be found locally at theatticusproject/cuad-qa, or remotely on the Hugging Face Hub.


In [5]:
cuad

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 22450
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 4182
    })
})

In [6]:
tokenizer = AutoTokenizer.from_pretrained("unsloth/Llama-3.2-1B-Instruct")

In [42]:
def build_split_by_length(dataset, min_length=0, max_length=2048, val_percentage=0.1, filter_ns=False):
    if filter_ns:
        buf_data = dataset.filter(lambda example: len(example['answers']['text']) != 0)
        buf_data = buf_data.filter(lambda example: np.logical_and(len(example["tokens"]) >= min_length, len(example["tokens"]) < max_length))
    else:
        # filter samples by length
        buf_data = dataset.filter(lambda example: np.logical_and(len(example["tokens"]) >= min_length, len(example["tokens"]) < max_length))
    # split valdation from train, but check that context does not overlap between train and val
    texts, counts = np.unique(buf_data["train"]["context"], return_counts=True)
    val_size = int(val_percentage * len(buf_data["train"]))
    val_len = 0
    val_texts = []
    for idx in range(len(texts)):
        val_texts.append(texts[idx])
        val_len += counts[idx]
        if val_len >= val_size:
            break
    val_data = buf_data["train"].filter(lambda example: example["context"] in val_texts)
    train_data = buf_data["train"].filter(lambda example: example["context"] not in val_texts)
    buf_data["validation"] = val_data
    buf_data["train"] = train_data
    return buf_data

In [8]:
# tokenize texts to calc length
def tokenize(el):
    if len(el['answers']['text']) != 0:
        answer = el['answers']['text'][0]
    else:
        answer = "Not stated."
    messages = [
        {"role": "user", "content": f"Question:\n{el['question']}\nContext:\n{el['context']}\n"},
        {"role": "assistant", "content": f"Answer:\n{answer}\n"},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )
    return {"tokens": text}

In [9]:
cuad["train"][11]["answers"]["text"]

[]

In [10]:
tok_cuad = cuad.map(tokenize)

In [16]:
b1 = build_split_by_length(tok_cuad)

Filter:   0%|          | 0/2314 [00:00<?, ? examples/s]

In [17]:
b1

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
        num_rows: 2050
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
        num_rows: 656
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
        num_rows: 264
    })
})

In [18]:
b1.save_to_disk("../../data/cuad_prepared/cuad_0k_2k")

Saving the dataset (0/1 shards):   0%|          | 0/2050 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/656 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/264 [00:00<?, ? examples/s]

In [20]:
start_size = [0, 2048, 4096, 8192, 16384]
end_size = [2048, 4096, 8192, 16384, 32768]
for idx in range(len(start_size)):
    buf_data = build_split_by_length(tok_cuad, start_size[idx], end_size[idx])
    save_path = f"../../data/cuad_prepared/cuad_{start_size[idx]//1000}k_{end_size[idx]//1000}k"
    print(save_path)
    buf_data.save_to_disk(save_path)

../../data/cuad_prepared/cuad_0k_2k


Saving the dataset (0/1 shards):   0%|          | 0/2050 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/656 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/264 [00:00<?, ? examples/s]

Filter:   0%|          | 0/22450 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/3922 [00:00<?, ? examples/s]

../../data/cuad_prepared/cuad_16k_32k


Saving the dataset (0/2 shards):   0%|          | 0/3469 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/494 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/453 [00:00<?, ? examples/s]

In [21]:
start_size = [0, 0, 0, 0, 0]
end_size = [2048, 4096, 8192, 16384, 32768]
for idx in range(len(start_size)):
    buf_data = build_split_by_length(tok_cuad, start_size[idx], end_size[idx])
    save_path = f"../../data/cuad_prepared/cuad_{start_size[idx]//1000}k_{end_size[idx]//1000}k"
    print(save_path)
    buf_data.save_to_disk(save_path)

../../data/cuad_prepared/cuad_0k_2k


Saving the dataset (0/1 shards):   0%|          | 0/2050 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/656 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/264 [00:00<?, ? examples/s]

Filter:   0%|          | 0/22450 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/20345 [00:00<?, ? examples/s]

../../data/cuad_prepared/cuad_0k_32k


Saving the dataset (0/7 shards):   0%|          | 0/18276 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/3977 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2069 [00:00<?, ? examples/s]

In [ ]:
# TODO: maybe build datasets for 3 and 5 segments instead of 4s

In [43]:
# Now filter all ns samples to more real dataset
start_size = [0, 2048, 4096, 8192, 16384]
end_size = [2048, 4096, 8192, 16384, 32768]
for idx in range(len(start_size)):
    buf_data = build_split_by_length(tok_cuad, start_size[idx], end_size[idx], filter_ns=True)
    save_path = f"../../data/cuad_prepared/cuad_no_ns_{start_size[idx]//1000}k_{end_size[idx]//1000}k"
    print(save_path)
    buf_data.save_to_disk(save_path)

Filter:   0%|          | 0/11180 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1244 [00:00<?, ? examples/s]

Filter:   0%|          | 0/470 [00:00<?, ? examples/s]

Filter:   0%|          | 0/470 [00:00<?, ? examples/s]

../../data/cuad_prepared/cuad_no_ns_0k_2k


...

Filter:   0%|          | 0/2605 [00:00<?, ? examples/s]

../../data/cuad_prepared/cuad_no_ns_16k_32k


Saving the dataset (0/1 shards):   0%|          | 0/2281 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/228 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/324 [00:00<?, ? examples/s]

In [44]:
d1 = datasets.load_from_disk("../../data/cuad_prepared/cuad_no_ns_0k_2k")

In [45]:
len(d1["train"]["answers"][5]["text"])

1

In [46]:
start_size = [0, 0, 0, 0, 0]
end_size = [2048, 4096, 8192, 16384, 32768]
for idx in range(len(start_size)):
    buf_data = build_split_by_length(tok_cuad, start_size[idx], end_size[idx], filter_ns=True)
    save_path = f"../../data/cuad_prepared/cuad_no_ns_{start_size[idx]//1000}k_{end_size[idx]//1000}k"
    print(save_path)
    buf_data.save_to_disk(save_path)

../../data/cuad_prepared/cuad_no_ns_0k_2k


Saving the dataset (0/1 shards):   0%|          | 0/421 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/71 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/49 [00:00<?, ? examples/s]

Filter:   0%|          | 0/11180 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/9573 [00:00<?, ? examples/s]

../../data/cuad_prepared/cuad_no_ns_0k_32k


Saving the dataset (0/3 shards):   0%|          | 0/8610 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1129 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/963 [00:00<?, ? examples/s]

In [47]:
buf_data

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
        num_rows: 8610
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
        num_rows: 1129
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
        num_rows: 963
    })
})

In [99]:
val, counts = np.unique(b1["train"]["context"], return_counts=True)

In [100]:
counts

array([45, 42, 44, 44, 44, 45, 46, 46, 45, 45, 44, 46, 44, 41, 45, 46, 45,
       44, 45, 42, 45, 51, 44, 44, 44, 46, 45, 42, 45, 44, 38, 48, 45, 44,
       50, 43, 50, 44, 43, 45, 43, 38, 44, 45, 44, 44, 44, 44, 45, 47, 44,
       44])

In [ ]:
len(counts)

In [ ]:
_, counts_all = np.unique(tok_cuad["train"]["context"], return_counts=True)

In [ ]:
len(counts_all)

In [ ]:
counts_all

In [36]:
tokens = tokenizer(cuad["train"]["context"])

In [38]:
lengths = [len(el) for el in tokens["input_ids"]]

In [52]:
np.min(lengths), np.max(lengths), np.mean(lengths), np.percentile(lengths,20), np.percentile(lengths,80), np.percentile(lengths,90)

(np.int64(235),
 np.int64(80233),
 np.float64(13408.608062360801),
 np.float64(3375.0),
 np.float64(22305.0),
 np.float64(32339.0))

In [63]:
np.sum(np.array(lengths) < 2048), np.sum(np.logical_and(np.array(lengths) < 4096, np.array(lengths) > 2048)),\
np.sum(np.logical_and(np.array(lengths) < 8192, np.array(lengths) > 4096)),\
np.sum(np.logical_and(np.array(lengths) < 16384, np.array(lengths) > 8192)),\
np.sum(np.logical_and(np.array(lengths) < 32768, np.array(lengths) > 16384))

(np.int64(2383),
 np.int64(3419),
 np.int64(4897),
 np.int64(5771),
 np.int64(3877))

In [45]:
cuad["train"][2]

{'id': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Parties_1',
 'title': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT',
 'context': 'EXHIBIT 10.6\n\n                              DISTRIBUTOR AGREEMENT\n\n         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Company")  and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999.\n\n                                    RECITALS\n\n         A. The  Company\'s  Business.  The Company is  presently  engaged in the business  of selling an energy  efficiency  device,  which is  referred to as an "Energy  Saver"  which may be improved  or  otherwise  changed  from its present composition (the "Products").  The Company may engage in the business of selling other  products  or  other  devices  other  than  the  Products,  which  will be considered  Products if Distributor  exercises its options pursuant to Section 7 hereof.\n\n     